In [6]:
! python main.py --epoches 1 -bs 8 -task polyp --exp_name example --split_path data/split.json --data_dir data/ -lr 0.01 --pos_rand_crop 0.85 --neg_rand_crop 0.15 --optim SGD --theashold_discretize 0.5 --image_key image --out_channels 2 --order 85 --criterion dice_ce --wandb_mode disabled

^C


In [ ]:
import torch
import torch.nn as nn
from segment_anything import sam_model_registry
from utils.utils import disable_batchnorm_running_stats
from utils.dataset_utils import get_train_val_dataloaders
from utils.model_utils import get_model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
sam_ckpt = "cp/" + 'sam_vit_b' + ".pth"
print(f"Loading SAM model from checkpoint: {sam_ckpt}")
sam = sam_model_registry['vit_b'](checkpoint=sam_ckpt)
sam.to(device=device)
img_dim = sam.image_encoder.img_size
disable_batchnorm_running_stats(sam)
print(f"Image dimension for SAM model: {img_dim}")

Using device: cpu
Loading SAM model from checkpoint: cp/sam_vit_b.pth
[INFO] disable_batchnorm_running_stats: updated 0 BatchNorm layer(s) to track_running_stats=False.
Image dimension for SAM model: 1024


In [ ]:
# ============================================================
# args for testing get_train_val_dataloaders
# ============================================================
import os
import sys
from pathlib import Path
import torch

# --- Project root (Windows) ---
PROJECT_ROOT = Path(r'C:\Users\Reza\Desktop\MT\polyp_sam_less')
# ← مسیر واقعی خود را جایگزین کنید

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("=" * 60)

# ============================================================
# args — minimal set needed by get_train_val_dataloaders
# ============================================================
args = {
    # --- task / paths ---
    'task':       'polyp',
    'split_path': str(PROJECT_ROOT / 'data' / 'split.json'),

    # --- dataset ---
    'n_fold':     0,
    'cache_rate': 0.0,           # no caching (safe for Windows RAM)
    'image_key':  'image',
    'mask_key':   'mask',

    # --- DataLoader ---
    'Batch_size': 2,
    'nW':         0,             # num_workers for train (0 = safe on Windows)
    'nW_eval':    0,             # num_workers for val

    # --- device ---
    'device':     device,
}

print("args:")
for k, v in args.items():
    print(f"  {k}: {v}")
print("=" * 60)

Device: cpu
args:
  task: polyp
  split_path: C:\Users\Reza\Desktop\MT\polyp_sam_less\data\split.json
  n_fold: 0
  cache_rate: 0.0
  image_key: image
  mask_key: mask
  Batch_size: 2
  nW: 0
  nW_eval: 0
  device: cpu


In [ ]:
train_dataloader, val_dataloader = get_train_val_dataloaders(args, img_dim)
img_ch = train_dataloader.dataset[0][args['image_key']].shape[0]

In [ ]:
# ============================================================
# args for testing get_model(args, sam, img_ch)
# ============================================================
import os
import sys
from pathlib import Path
import torch

# --- Project root (Windows) ---
PROJECT_ROOT = Path(r'C:\Users\Reza\Desktop\MT\polyp_sam_less')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("=" * 60)

# ============================================================
# args needed by get_model + ModelEmb + HarDNet
# ============================================================
args = {
    # --- get_model ---
    'model_emb':  'HardNet',      # selector in get_model
    'ckpt_path':  None,           # no pretrained prompt learner (Phase 1)
    'device':     device,         # torch.device

    # --- ModelEmb → HarDNet ---
    'depth_wise': False,          # HarDNet standard (not depthwise)
    'order':      68,             # HarDNet-68 (arch variant)
    'pretrained': True,           # ImageNet weights (already cached)
}

print("args:")
for k, v in args.items():
    print(f"  {k}: {v}")
print("=" * 60)

# --- img_ch for polyp ---
img_ch = 3
print(f"img_ch: {img_ch}")
print("=" * 60)

Device: cpu
args:
  model_emb: HardNet
  ckpt_path: None
  device: cpu
  depth_wise: False
  order: 68
  pretrained: True
img_ch: 3


In [ ]:
model = get_model(args, sam, img_ch)

ImageNet pretrained weights for HarDNet68 is loaded
